# 3. Predicción sobre test
Aplica el modelo final (entrenado y guardado por `2_entrenar_modelo_final.ipynb`) a los datos de test y genera el archivo para la competencia.

**Entradas:** los modelos de `modelos/`, los datos de la competencia, `preds/test_*` (modelos de lenguaje) y `llm_dudosas_scores_v2.csv`.

**Salida:** `submission.csv`.

In [1]:
import os, pickle, time, warnings
import numpy as np, pandas as pd
from tp3_utils import *
from tp3_utils import _logit
warnings.filterwarnings('ignore')

def buscar(nombre):                      # los .jsonlines pueden estar en esta carpeta o en la de arriba
    for p in [nombre, os.path.join('..', nombre)]:
        if os.path.exists(p): return p
    raise FileNotFoundError(nombre)
TRAIN, TEST = buscar('train_data.jsonlines'), buscar('test_data.jsonlines')
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

### Variables de test
Mismas funciones que en el entrenamiento (leen el caché que dejó la notebook 2).

In [2]:
d = cargar_todo(TRAIN, TEST)             # variables de la publicación, del vendedor y de la categoría
raw = cargar_raw(TRAIN, TEST)            # datos crudos aplanados
y = d['y'].values                        # 1 = usado, 0 = nuevo

# XLM-R: su probabilidad de 'usado' entra como una variable más (en logit, igual que BERT y BETO)
oof_xlmr, test_xlmr = np.load('preds/oof_xlmr.npy'), np.load('preds/test_xlmr.npy')
def con_xlmr(F, X, Xt):
    F = [(a, b, Xa.assign(xlmr=_logit(oof_xlmr[a])), np.asarray(ya), Xb.assign(xlmr=_logit(oof_xlmr[b])), np.asarray(yb)) for a, b, Xa, ya, Xb, yb in F]
    return F, X.assign(xlmr=_logit(oof_xlmr)), Xt.assign(xlmr=_logit(test_xlmr))

# LightGBM / XGBoost: set completo con target encodings calculados dentro de cada fold
F_T, X_T, Xt_T = con_xlmr(*armar_final_tv(d, raw, te=True, raw_cats=False, base='v3'))
# CatBoost: recibe seller_id y category_id como categóricas nativas (sin target encodings a mano)
F_C, X_C, Xt_C = con_xlmr(*armar_final_tv(d, raw, te=False, raw_cats=True, base='v2'))
CATC = list(CATS) + ['cat_seller_id', 'cat_category_id']
def prep(X):
    X = X.copy()
    for c in CATC: X[c] = X[c].astype(str)
    return X
F_C = [(a, b, prep(Xa), ya, prep(Xb), yb) for a, b, Xa, ya, Xb, yb in F_C]; X_C, Xt_C = prep(X_C), prep(Xt_C)
print('LightGBM / XGBoost:', X_T.shape[1], 'variables | CatBoost:', X_C.shape[1], 'variables')

LightGBM / XGBoost: 133 variables | CatBoost: 126 variables


### Predicción de los 3 modelos
Cada algoritmo promedia sus semillas; después se promedian los 3.

In [3]:
def xgb_cargar(path):
    m = XGBClassifier(enable_categorical=True); m.load_model(path); return m
PRED = {'lgbm': lambda path, X: lgb.Booster(model_file=path).predict(X),
        'xgb':  lambda path, X: xgb_cargar(path).predict_proba(X)[:, 1],
        'cat':  lambda path, X: CatBoostClassifier().load_model(path).predict_proba(X)[:, 1]}
EXT = {'lgbm': 'txt', 'xgb': 'json', 'cat': 'cbm'}
def predecir(nombre, X):
    archivos = sorted(f for f in os.listdir('modelos') if f.startswith(nombre + '_s') and f.endswith(EXT[nombre]))
    print(nombre, archivos)
    return np.mean([PRED[nombre]('modelos/' + f, X) for f in archivos], 0)
P = {'lgbm': predecir('lgbm', Xt_T), 'xgb': predecir('xgb', Xt_T), 'cat': predecir('cat', Xt_C)}
p_ens = np.mean(list(P.values()), 0)
print('ensamble:', p_ens.shape, '| % usado:', (p_ens >= .5).mean().round(4))

lgbm ['lgbm_s123.txt', 'lgbm_s42.txt', 'lgbm_s7.txt']


xgb ['xgb_s42.json', 'xgb_s7.json']


cat ['cat_s42.cbm']
ensamble: (30000,) | % usado: 0.4731


### Ponderación con el LLM en las publicaciones dudosas
Se aplica la regresión logística guardada a las filas de test que tienen puntaje del LLM; el resto queda con la probabilidad del ensamble.

In [4]:
# Los puntajes del LLM son probabilidades entre 0 y 1; se pasan a logit (recortando a [0.02, 0.98])
logit_llm = lambda q: np.log(np.clip(q, .02, .98) / (1 - np.clip(q, .02, .98)))
def entradas_llm(p_modelo, S, cols):
    return np.column_stack([_logit(p_modelo[S.fila.values])] + [logit_llm(S[c].values) for c in cols])

PONDERACION = pickle.load(open('modelos/ponderacion_llm.pkl', 'rb'))
S = pd.read_csv('llm_dudosas_scores_v2.csv'); S = S[S.base == 'test']
nucleo = S.p_fuerte.notna()
p_final = p_ens.copy()
for nombre, filas in [('banda', S[~nucleo]), ('nucleo', S[nucleo])]:
    modelo, cols = PONDERACION[nombre]
    p_final[filas.fila.values] = modelo.predict_proba(entradas_llm(p_ens, filas, cols))[:, 1]
print('filas con LLM:', len(S), '| predicciones que cambian de lado:', ((p_final >= .5) != (p_ens >= .5)).sum())

filas con LLM: 2759 | predicciones que cambian de lado: 359


### Archivo para Kaggle

In [5]:
sub = guardar_submission(d, p_final, 'submission.csv')
print(sub.shape, sub.condition.value_counts().to_dict())
sub.head()

(30000, 2) {'new': 15758, 'used': 14242}


,ID,condition
0,1,used
1,2,new
2,3,used
3,4,used
4,5,new
